# Experiment 19: GAN for Generating Handwritten Digits using MNIST

**Objective:** Build and train a Generative Adversarial Network (GAN) to generate handwritten digit images using the MNIST dataset.

**Language:** Python  
**Estimated Time:** 2 hours  
**Libraries Used:** NumPy, Matplotlib, TensorFlow / Keras


## 1. Introduction

A **Generative Adversarial Network (GAN)** consists of two neural networks:

- **Generator:** Creates fake images from random noise.
- **Discriminator:** Tries to distinguish real images from fake images.

Both networks compete with each other during training. Over time, the generator learns to create images that look more like real handwritten digits.


## 2. Aim of the Experiment

In this notebook, we will:

1. Load and preprocess the MNIST dataset
2. Build the generator model
3. Build the discriminator model
4. Combine them into a GAN
5. Train the GAN
6. Generate handwritten digit images
7. Visualize training progress and generated outputs


In [ ]:
# Install if needed
# !pip install tensorflow matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Dense, Reshape, Flatten, LeakyReLU, Dropout, BatchNormalization, Input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

## 3. Load and Preprocess MNIST Dataset

MNIST contains 28x28 grayscale images of handwritten digits from 0 to 9. We normalize pixel values to the range **[-1, 1]** because the generator uses `tanh` activation in the output layer.


In [ ]:
(x_train, _), (_, _) = mnist.load_data()

x_train = x_train.astype('float32')
x_train = (x_train - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)

print('Training data shape:', x_train.shape)
print('Pixel range:', x_train.min(), 'to', x_train.max())

In [ ]:
plt.figure(figsize=(8, 3))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
    plt.title('Real')
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Define Hyperparameters


In [ ]:
latent_dim = 100
img_rows = 28
img_cols = 28
channels = 1
img_shape = (img_rows, img_cols, channels)

batch_size = 128
epochs = 3000
sample_interval = 500

## 5. Build the Generator

The generator converts a random noise vector into a 28x28 image.


In [ ]:
def build_generator(latent_dim):
    model = Sequential(name='Generator')
    model.add(Input(shape=(latent_dim,)))
    model.add(Dense(256))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(512))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(1024))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(np.prod(img_shape), activation='tanh'))
    model.add(Reshape(img_shape))
    return model

generator = build_generator(latent_dim)
generator.summary()

## 6. Build the Discriminator

The discriminator receives an image and predicts whether it is real or fake.


In [ ]:
def build_discriminator(img_shape):
    model = Sequential(name='Discriminator')
    model.add(Input(shape=img_shape))
    model.add(Flatten())
    model.add(Dense(512))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Dropout(0.3))
    model.add(Dense(256))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Dropout(0.3))
    model.add(Dense(1, activation='sigmoid'))
    return model

discriminator = build_discriminator(img_shape)
discriminator.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])
discriminator.summary()

## 7. Build the Combined GAN Model

The discriminator is frozen while training the combined GAN so that only the generator learns to fool it.


In [ ]:
z = Input(shape=(latent_dim,))
img = generator(z)

discriminator.trainable = False
validity = discriminator(img)

gan = Model(z, validity, name='GAN')
gan.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))
gan.summary()

## 8. Helper Function to Save Generated Images


In [ ]:
def plot_generated_images(epoch, generator, latent_dim, examples=16, dim=(4, 4), figsize=(6, 6)):
    noise = np.random.normal(0, 1, (examples, latent_dim))
    gen_imgs = generator.predict(noise, verbose=0)
    gen_imgs = 0.5 * gen_imgs + 0.5  # scale from [-1,1] to [0,1]

    plt.figure(figsize=figsize)
    for i in range(gen_imgs.shape[0]):
        plt.subplot(dim[0], dim[1], i + 1)
        plt.imshow(gen_imgs[i, :, :, 0], cmap='gray')
        plt.axis('off')
    plt.suptitle(f'Generated Images at Epoch {epoch}', y=1.02)
    plt.tight_layout()
    plt.show()

## 9. Train the GAN

In each training step:

1. Train the discriminator on real and fake images
2. Train the generator through the combined GAN model


In [ ]:
d_losses = []
g_losses = []

valid = np.ones((batch_size, 1))
fake = np.zeros((batch_size, 1))

for epoch in range(1, epochs + 1):
    # ---------------------
    #  Train Discriminator
    # ---------------------
    idx = np.random.randint(0, x_train.shape[0], batch_size)
    real_imgs = x_train[idx]

    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    gen_imgs = generator.predict(noise, verbose=0)

    discriminator.trainable = True
    d_loss_real = discriminator.train_on_batch(real_imgs, valid)
    d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # -----------------
    #  Train Generator
    # -----------------
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    discriminator.trainable = False
    g_loss = gan.train_on_batch(noise, valid)

    d_losses.append(float(d_loss[0]))
    g_losses.append(float(g_loss if np.isscalar(g_loss) else g_loss[0]))

    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{epochs}  [D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.2f}%]  [G loss: {float(g_loss if np.isscalar(g_loss) else g_loss[0]):.4f}]")

    if epoch % sample_interval == 0:
        plot_generated_images(epoch, generator, latent_dim)

## 10. Plot Generator and Discriminator Loss


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(d_losses, label='Discriminator Loss')
plt.plot(g_losses, label='Generator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Loss')
plt.legend()
plt.grid(True)
plt.show()

## 11. Generate Final Handwritten Digit Images


In [ ]:
plot_generated_images('Final', generator, latent_dim, examples=25, dim=(5, 5), figsize=(7, 7))

## 12. Conclusion

In this experiment, we built and trained a GAN on the MNIST dataset to generate handwritten digits. The generator learned to create images from random noise, while the discriminator learned to classify real and fake images. After training, the GAN was able to produce digit-like images that resemble handwritten digits.


## 13. Viva Questions

1. What is the role of the generator in a GAN?
2. What is the role of the discriminator?
3. Why is `tanh` used in the generator output layer?
4. Why are MNIST images normalized to [-1, 1]?
5. What happens if the discriminator becomes too strong?
